In [ ]:
import os
import numpy as np
import pandas as pd


os.chdir(r'D:\Work\景观格局指标对PM2.5影响')

# 数据读取
df = pd.read_excel("指标和气象因子数据（分LCZ类型）.xlsx", index_col=0)

In [ ]:
df = df[df["lcz_type"] <= 9]

In [ ]:
df["lcz_type"].unique()

In [ ]:
# sklearn库和shap库调用
from sklearn.model_selection import cross_validate
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split

import shap
shap.initjs()


# 数据拆分为特征数据和目标数据
data = df.iloc[:, 1:-4]
targets = df.iloc[:, -4:]
target_names = targets.columns

In [ ]:
from sklearn.metrics import r2_score

In [ ]:
for target_name in target_names:
    # target_name = "winter"
    target = df[target_name]
    feature_name = data.columns[7:].tolist() + data.columns[data.columns.str.contains(target_name)].tolist()
    feature_data = df[feature_name]
    
    # 选择80%训练数据和20%测试数据
    X_train, X_test, y_train, y_test = train_test_split(
        feature_data, target, train_size=0.8, test_size=0.2, random_state=925) 

    gbr = GradientBoostingRegressor(
        learning_rate=0.01,
        n_estimators=2000,
        n_iter_no_change=10,
        validation_fraction=0.1,
        tol = 0.000001,
        max_depth=3,
        random_state=925,
    )
    gbr.fit(X_train, y_train)

    print("最佳拟合树数量：", gbr.n_estimators_)
    
    scores = cross_val_score(
        gbr,
        X_test, y_test,
        cv=5, scoring='neg_mean_squared_error'
    )
    print(target_name, "RMSE: ", np.sqrt(-scores.mean()))
    y_pred = gbr.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    print(target_name, "R-squared:", r2)

    

In [ ]:
for target_name in target_names:
    # target_name = "winter"
    target = df[target_name]
    feature_name = data.columns[7:].tolist()
    feature_data = df[feature_name]
    
    # 选择80%训练数据和20%测试数据
    X_train, X_test, y_train, y_test = train_test_split(
        feature_data, target, train_size=0.8, test_size=0.2, random_state=925) 

    gbr = GradientBoostingRegressor(
        learning_rate=0.01,
        n_estimators=500,
        n_iter_no_change=10,
        tol = 0.000001,
        max_depth=3,
        random_state=925,
    )
    gbr.fit(X_train, y_train)
    
    scores = cross_val_score(
        gbr,
        X_test, y_test,
        cv=5, scoring='neg_mean_squared_error'
    )
    print(target_name, "RMSE: ", np.sqrt(-scores.mean()))
    y_pred = gbr.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    print(target_name, "R-squared:", r2)

In [ ]:
target_names

In [ ]:
# 选取春季数据作为模型目标值
target_name = "spring"
target = df[target_name]
feature_name = data.columns[7:].tolist() + data.columns[data.columns.str.contains(target_name)].tolist()
feature_data = df[feature_name]

# 选择80%训练数据和20%测试数据
X_train, X_test, y_train, y_test = train_test_split(
    feature_data, target, train_size=0.8, test_size=0.2, random_state=925) 

In [ ]:
feature_data

In [ ]:
# 使用固定超参数和早停机制训练模型并计算RMSE
gbr = GradientBoostingRegressor(
    learning_rate=0.01,
    n_estimators=2000,
    n_iter_no_change=10,
    tol = 0.000001,
    max_depth=3,
    random_state=925,
)
gbr.fit(X_train, y_train)

scores = cross_val_score(
    gbr,
    X_test, y_test,
    cv=5, scoring='neg_mean_squared_error'
)
print("RMSE: ", np.sqrt(-scores.mean()))

In [ ]:
# 使用SHAP分析解释训练后的模型
explainer = shap.Explainer(gbr)
shap_values = explainer(X_test)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import string
from statsmodels.nonparametric.smoothers_lowess import lowess


config = {
    "font.family": 'serif',
    "font.size": 12,
    "mathtext.fontset": 'stix',
    "font.serif": 'Times New Roman',
    "axes.unicode_minus": False,
}
plt.rcParams.update(config)

In [ ]:
# 所有样本的SHAP值展示,并按重要性排序
fig, ax = plt.subplots(figsize = (8, 6))
shap.plots.beeswarm(shap_values, plot_size=None, ax=ax, max_display=20, show=False)
ax.set_title(target_name, x = 0.5, y = 1, fontsize=15)
fig.savefig(f"SHAP重要性图({target_name}).png", dpi=300, bbox_inches='tight')

In [ ]:
# 特征重要性条形图
fig, ax = plt.subplots(figsize = (8, 6))
shap.plots.bar(shap_values, ax=ax, max_display=20, show=False)
ax.set_title(target_name, x = 0.5, y = 1, fontsize=15)
fig.savefig(f"SHAP重要性条形图({target_name}).png", dpi=300, bbox_inches='tight')

In [ ]:
# 获取模型重要性特征序列
shap_importance = pd.DataFrame({
    "feature": X_test.columns,
    "importance": np.abs(shap_values.values).mean(axis=0)
}).sort_values("importance", ascending=False)

shap_max_importance_features = shap_importance['feature'].tolist()

In [ ]:
# 按模型特征重要性顺序进行出图, 并使用局部加权散点平滑来进行非线性拟合

fig = plt.figure(figsize = (32, 24))
grid = GridSpec(4, 4)

labels = [f'({letter})' for letter in string.ascii_lowercase[:len(shap_max_importance_features)]]

for i, feature in enumerate(shap_max_importance_features):
    grid_x = i // 4
    grid_y = i % 4
    ax = fig.add_subplot(grid[grid_x, grid_y])
    ax.set_facecolor("#e3e5ff")
    ax.grid(color='#FFFFFF', linewidth=2)
    cb = shap.dependence_plot(feature, shap_values.values, X_test, ax=ax, show=False)
    ax.set_ylabel("SHAP Value")
    ax.set_axisbelow(True)
   
    for spine in ax.spines.values():
        spine.set_visible(False)

    # 隐藏刻度线但保留标签
    ax.tick_params(axis='both', 
                   which='both',
                   length=0,      # 刻度线长度为0
                   width=0,       # 刻度线宽度为0
                   bottom=True,   # 保留底部标签
                   left=True,     # 保留左侧标签
                   labelbottom=True, 
                   labelleft=True)
    cb = plt.gcf().axes[-1]
    cb.tick_params(length=0)
    ax.text(0, 1.07, labels[i], 
            transform=ax.transAxes,
            fontsize=14,
            va='top', ha='left',
            bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))

    # 非线性拟合线
    x = X_test[feature]
    y = shap_values.values[:, X_test.columns.get_loc(feature)]
    lowess_smoothed = lowess(y, x, frac=0.5)
    ax.plot(lowess_smoothed[:, 0], lowess_smoothed[:, 1], 'r-', label='Trend line')
# 保存图片
fig.suptitle(target_name, x = 0.5, y = 0.92, fontsize=28)
fig.savefig(f"主要的单特征依赖图({target_name}).png", dpi=300, bbox_inches='tight')

In [ ]:
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import os
from matplotlib import rcParams

In [ ]:
config = {
    "font.family": 'serif',
    "font.size": 12,
    "mathtext.fontset": 'stix',
    "font.serif": 'Simsun',
}
rcParams.update(config)

In [ ]:
target_name = target_names[0]
target = df[target_name]

In [ ]:
for i in data:
    com = (data[i] != 0) & (target != 0)
    data1 = data[i][com]
    data2 = target[com]
    OLS = stats.linregress(data1, data2)
    print(f"{i}-----slope: {OLS[0]}, r: {OLS[2]}, p_value: {OLS[3]:.5f}")

In [ ]:
data2

In [ ]:
for i in data:
    com = (data[i] != 0) & (target != 0)
    data1 = data[i][com]
    data2 = target[com]
    OLS = stats.linregress(data[i], target)
    print(f"{i}-----slope: {OLS[0]}, r2: {OLS[2]}, p_value: {OLS[3]}")

In [ ]:
s

In [ ]:
r**2

In [ ]:
p